# Selective Prediction — Clinician Referral Safety Layer

> **CLINICAL DECISION SUPPORT ONLY — NOT AN AUTONOMOUS DIAGNOSTIC SYSTEM.**
> This notebook builds a referral layer that routes the model's least-confident predictions to **mandatory clinician review**. Predictions for "accepted" (non-referred) cases are decision-support inputs for a clinician to weigh alongside the full clinical picture — never a diagnosis, and never a reason to skip clinician review. "Referred" cases require full, standard-of-care clinician review because model confidence was assessed as too low to support even an advisory output. **No referral-rate setting makes this system suitable for unsupervised or autonomous operation.** See `src/selective_prediction.py`'s `CLINICAL_DISCLAIMER` for the canonical statement.

For each of the three calibrated baselines: rank validation cases by predictive-entropy uncertainty, test referral rates of 0/5/10/20/30%, and compute coverage, selective risk, accepted-case error rate, sensitivity, specificity, PPV, NPV, Brier score, and ECE on the accepted (non-referred) subset at each level.

**This notebook uses validation-set predictions only — the test split is loaded for deterministic split reproduction and immediately discarded, consistent with every prior notebook in this project.**

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch

from src.config import load_config
from src.reproducibility import set_global_seed, get_logger, log_run_metadata
from src.preprocess import load_and_split_cohort, drop_identifier_columns, IDENTIFIER_COLUMNS
from src.models import MLPClassifierTorch
from src.calibration import PlattScaler, IsotonicCalibrator, TemperatureScaler
from src.selective_prediction import (
    CLINICAL_DISCLAIMER,
    evaluate_referral_policy,
    select_best_calibration_method,
)
from src.plots import (
    plot_risk_coverage_curve,
    plot_accuracy_coverage_curve,
    plot_calibration_vs_coverage_curve,
)

print(CLINICAL_DISCLAIMER)

config = load_config()
seed = config["project"]["random_seed"]
set_global_seed(seed)
logger = get_logger(log_file=config["logging"]["log_file"])
logger.info("Selective prediction notebook started (seed=%d)", seed)
logger.info("CLINICAL FRAMING: decision-support referral layer only, not autonomous diagnosis.")

TARGET_COL = config["preprocessing"]["target_column"]
TRAIN_MODELS_DIR = Path(config["training"]["models_output_dir"])
CAL_MODELS_DIR = Path(config["calibration"]["models_output_dir"])
SP_CONFIG = config["selective_prediction"]
REFERRAL_RATES = SP_CONFIG["referral_rates"]
THRESHOLD = SP_CONFIG["decision_threshold"]
ECE_BINS = SP_CONFIG["ece_bins"]

## 1. Reproduce the split, build X_val

Same deterministic split + preprocessor as `02_train_models.ipynb` / `03_calibration.ipynb`. Test split loaded and immediately discarded.

In [ ]:
train_df, val_df, test_df = load_and_split_cohort(config)
del test_df
del train_df

preprocessor = joblib.load(config["preprocessing"]["preprocessor_output_path"])
val_features_df = drop_identifier_columns(val_df, IDENTIFIER_COLUMNS)
X_val = preprocessor.transform(val_features_df)
y_val = val_features_df[TARGET_COL].reset_index(drop=True).to_numpy()

print(f"X_val: {X_val.shape}, positive rate: {y_val.mean():.3f}")

## 2. Load trained models, recompute uncalibrated validation predictions

In [ ]:
logreg_model = joblib.load(TRAIN_MODELS_DIR / "logistic_regression.joblib")
lgbm_model = joblib.load(TRAIN_MODELS_DIR / "lightgbm.joblib")

mlp_checkpoint = torch.load(TRAIN_MODELS_DIR / "mlp_torch.pt", weights_only=True)
mlp_model = MLPClassifierTorch(
    input_dim=mlp_checkpoint["input_dim"], hidden_dims=mlp_checkpoint["hidden_dims"], seed=seed
)
mlp_model.load_state_dict(mlp_checkpoint["state_dict"])
mlp_model.eval()

raw_prob = {
    "logistic_regression": logreg_model.predict_proba(X_val)[:, 1],
    "lightgbm": lgbm_model.predict_proba(X_val)[:, 1],
}
with torch.no_grad():
    mlp_logits = mlp_model(torch.tensor(X_val.to_numpy(dtype=np.float32))).squeeze().numpy()
raw_prob["mlp_torch"] = 1.0 / (1.0 + np.exp(-mlp_logits))

## 3. Select each model's best calibration method (data-driven) and apply it

Picks, per model, whichever fitted calibrator (from `03_calibration.ipynb`) achieved the lowest validation `ece_10bins` in `table_3_calibration_metrics.csv` — not a hardcoded choice.

In [ ]:
calibration_metrics_df = pd.read_csv(config["calibration"]["metrics_table_path"])

calibrated_prob = {}
calibration_method_used = {}

for model_name in ["logistic_regression", "lightgbm", "mlp_torch"]:
    candidates = SP_CONFIG["calibration_candidates"][model_name]
    best_method = select_best_calibration_method(calibration_metrics_df, model_name, candidates)
    calibration_method_used[model_name] = best_method

    if best_method == "temperature":
        checkpoint = torch.load(CAL_MODELS_DIR / f"calibration_{model_name}_temperature.pt", weights_only=True)
        scaler = TemperatureScaler()
        scaler.load_state_dict(checkpoint["state_dict"])
        calibrated_prob[model_name] = scaler.transform(mlp_logits)
    else:
        scaler = joblib.load(CAL_MODELS_DIR / f"calibration_{model_name}_{best_method}.joblib")
        calibrated_prob[model_name] = scaler.transform(raw_prob[model_name])

    logger.info("[%s] selected calibration method: %s (lowest validation ECE-10)", model_name, best_method)
    print(f"{model_name}: best calibration method = {best_method}")

## 4. Evaluate the referral policy at each coverage level

Cases are ranked by predictive entropy of the CALIBRATED probability (highest entropy = most uncertain = referred first). All metrics below (other than coverage/n_accepted/n_referred) are computed on the accepted (non-referred) subset only — this is a decision-support routing exercise, not a claim that accepted-case predictions can be used unsupervised.

In [ ]:
policy_frames = []
for model_name, y_prob in calibrated_prob.items():
    policy_df = evaluate_referral_policy(
        y_val, y_prob, referral_rates=REFERRAL_RATES, threshold=THRESHOLD, ece_bins=ECE_BINS
    )
    policy_df.insert(0, "model", model_name)
    policy_df.insert(1, "calibration_method", calibration_method_used[model_name])
    policy_frames.append(policy_df)
    logger.info("[%s] referral policy evaluated at referral rates %s", model_name, REFERRAL_RATES)

referral_policy_df = pd.concat(policy_frames, ignore_index=True)

policy_path = Path(SP_CONFIG["referral_policy_table_path"])
policy_path.parent.mkdir(parents=True, exist_ok=True)
referral_policy_df.to_csv(policy_path, index=False)
logger.info("Clinical referral policy table saved to %s", policy_path)

referral_policy_df

## 5. Risk-coverage, accuracy-coverage, and calibration-vs-coverage curves

In [ ]:
FIGURES_DIR = Path(SP_CONFIG["figures_output_dir"])

plot_risk_coverage_curve(referral_policy_df, save_path=FIGURES_DIR / "risk_coverage_curve.png")
plot_accuracy_coverage_curve(referral_policy_df, save_path=FIGURES_DIR / "accuracy_coverage_curve.png")
plot_calibration_vs_coverage_curve(referral_policy_df, save_path=FIGURES_DIR / "calibration_vs_coverage_curve.png")

logger.info("Risk-coverage, accuracy-coverage, and calibration-vs-coverage curves saved to %s", FIGURES_DIR)
print(f"Saved 3 coverage curves to {FIGURES_DIR}")

## 6. Clinical referral policy table (formatted for review)

In [ ]:
display_cols = [
    "model", "calibration_method", "referral_rate", "coverage", "n_accepted", "n_referred",
    "selective_risk", "accepted_case_error_rate", "sensitivity", "specificity",
    "ppv", "npv", "brier_score", "ece",
]
referral_policy_df[display_cols].round(4)

## 7. Run metadata

> Reminder: **this referral layer is clinical decision support only.** It does not make autonomous diagnostic decisions at any coverage level, including 0% referral.

In [ ]:
log_run_metadata(
    output_path=SP_CONFIG["run_metadata_path"],
    seed=seed,
    extra={
        "step": "selective_prediction",
        "n_val": int(len(y_val)),
        "referral_rates": REFERRAL_RATES,
        "decision_threshold": THRESHOLD,
        "calibration_method_used": calibration_method_used,
        "clinical_framing": "decision_support_only_not_autonomous_diagnosis",
        "test_labels_used": False,
    },
)
logger.info("Selective prediction notebook finished")
print(CLINICAL_DISCLAIMER)